## 线性回归的简洁实现

In [51]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [52]:
true_w = torch.tensor([2,-3.4])
true_b = 4.2
features,labels = d2l.synthetic_data(true_w,true_b,1000)

features

tensor([[-0.7417,  0.2515],
        [ 0.3424,  1.4324],
        [-0.8539, -1.0084],
        ...,
        [ 1.0876, -0.8714],
        [ 1.6882,  1.8077],
        [-1.2882,  0.5758]])

## 读取数据集

我们可以调用现有的 api 来读取数据，并通过数据迭代器指定 `batch_size`，此外，`is_train` 表示是否希数据迭代器对象在每个迭代器周期内打乱数据

In [53]:
def load_array(data_arrays,batch_size,is_train=True):
    """构造一个 Pytorch 数据迭代器"""
    dataset = data.TensorDataset(*data_arrays) # 注意这里要给 data_array 解包 因为 data_array 可以是元组 传入的参数是可变参数
    return data.DataLoader(dataset,batch_size,shuffle=is_train)

In [54]:
batch_size = 10
data_iter = load_array((features,labels),batch_size)

In [55]:
next(iter(data_iter))

[tensor([[ 0.6378,  1.0738],
         [ 0.2189,  0.5237],
         [ 0.4639, -0.2746],
         [-0.8639,  0.1081],
         [-0.1550,  1.3176],
         [-1.4711, -1.6185],
         [-0.1940, -0.0710],
         [ 0.8360,  0.2127],
         [ 0.8025, -0.6591],
         [-1.2992,  0.1337]]),
 tensor([[ 1.8283],
         [ 2.8600],
         [ 6.0617],
         [ 2.1066],
         [-0.5965],
         [ 6.7619],
         [ 4.0587],
         [ 5.1248],
         [ 8.0492],
         [ 1.1442]])]

## 定义模型

在PyTorch中，全连接层在`Linear`类中定义。
值得注意的是，我们将两个参数传递到`nn.Linear`中。
第一个指定输入特征形状，即2，第二个指定输出特征形状，输出特征形状为单个标量，因此为1。

In [56]:
from torch import nn

net = nn.Sequential(nn.Linear(2,1))

## 初始化模型参数

In [57]:
net[0].weight.data.normal_(0,0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 定义损失函数

In [58]:
loss = nn.MSELoss()

## 定义优化算法

In [59]:
trainer = torch.optim.SGD(net.parameters(),lr=0.03)

# 训练模型

In [ ]:
num_epoch = 3
for epoch in range(num_epoch):
    for X,y in data_iter:
        l = loss(net(X),y)
        trainer.zero_grad()
        l.backward() # 底层会自动.sum() 不用显式调用
        trainer.step() # 根据梯度更新参数

    l = loss(net(features),labels)
    print(f'epoch {epoch + 1},loss {l:f}')


epoch 1,loss 0.000276
epoch 2,loss 0.000103
epoch 3,loss 0.000103


In [61]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([ 0.0004, -0.0009])
b的估计误差： tensor([-0.0001])
